In [1]:
import os
import logging
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from transformers import ViTFeatureExtractor
from transformers import ViTImageProcessor
import cv2
from tqdm import tqdm
import gc
from collections import defaultdict
from timm import create_model
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Clear any unused memory
gc.collect()
torch.cuda.empty_cache()
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)
import logging
import sys

# Configure logger to output to console with no buffering
logging.basicConfig(stream=sys.stdout, level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()

# Force logging to flush after each message
handler = logging.StreamHandler(sys.stdout)
handler.flush = sys.stdout.flush
logger.addHandler(handler)



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


In [2]:
import os
import pandas as pd
import random

# Define the base directory for the dataset
video_dir = '/kaggle/input/hmdbdata/hmdb51'

# Initialize lists for filenames, labels, and splits
filenames = []
labels = []
splits = []

# Get the list of all classes (subdirectories)
classes = [class_name for class_name in os.listdir(video_dir) if os.path.isdir(os.path.join(video_dir, class_name))]

# For each class, get all video files and assign splits
for class_name in classes:
    class_dir = os.path.join(video_dir, class_name)
    
    # Get all video files in the class directory
    video_files = [f for f in os.listdir(class_dir) if f.endswith('.avi')]
    
    # Shuffle the video files (optional)
    random.shuffle(video_files)
    
    # Assign splits (70% train, 15% val, 15% test)
    total_videos = len(video_files)
    train_count = int(0.7 * total_videos)  # 70% for training
    test_count = total_videos - train_count  # Remaining for testing (30%)
    val_count = test_count // 2  # Half of the testing count for validation
    
    # Assign splits
    for i, video_file in enumerate(video_files):
        video_path = os.path.join(class_dir, video_file)
        filenames.append(video_path)  # Full path to the video file
        labels.append(class_name)  # The class name as the label
        
        # Assign splits based on the index
        if i < train_count:
            splits.append('train')
        elif i < train_count + val_count:
            splits.append('val')
        else:
            splits.append('test')

# Create a new DataFrame with the columns: filename, label, split
df_new = pd.DataFrame({
    'filename': filenames,
    'label': labels,
    'split': splits
})

# Save the new CSV file
output_csv_path = '/kaggle/working/new_hmdb51.csv'
df_new.to_csv(output_csv_path, index=False)

# Print the first few rows to verify
print(f"New CSV file saved at {output_csv_path}")
print(df_new.head())


New CSV file saved at /kaggle/working/new_hmdb51.csv
                                            filename      label  split
0  /kaggle/input/hmdbdata/hmdb51/kick_ball/Britne...  kick_ball  train
1  /kaggle/input/hmdbdata/hmdb51/kick_ball/Ribery...  kick_ball  train
2  /kaggle/input/hmdbdata/hmdb51/kick_ball/GOAL_2...  kick_ball  train
3  /kaggle/input/hmdbdata/hmdb51/kick_ball/Amazin...  kick_ball  train
4  /kaggle/input/hmdbdata/hmdb51/kick_ball/Nike_S...  kick_ball  train


In [3]:
# Print a few sample paths to check
print("Sample paths in 'filename' column:")
print(df_new['filename'].head())

# Output the counts
train_videos = df_new[df_new['split'] == 'train']['filename'].count()
val_videos = df_new[df_new['split'] == 'val']['filename'].count()
test_videos = df_new[df_new['split'] == 'test']['filename'].count()

print(f"Total training videos: {train_videos}")
print(f"Total validation videos: {val_videos}")
print(f"Total testing videos: {test_videos}")

Sample paths in 'filename' column:
0    /kaggle/input/hmdbdata/hmdb51/kick_ball/Britne...
1    /kaggle/input/hmdbdata/hmdb51/kick_ball/Ribery...
2    /kaggle/input/hmdbdata/hmdb51/kick_ball/GOAL_2...
3    /kaggle/input/hmdbdata/hmdb51/kick_ball/Amazin...
4    /kaggle/input/hmdbdata/hmdb51/kick_ball/Nike_S...
Name: filename, dtype: object
Total training videos: 4714
Total validation videos: 1010
Total testing videos: 1042


In [4]:
import pandas as pd

# Load the new CSV file
df_hmdb51 = pd.read_csv('/kaggle/working/new_hmdb51.csv')

# Print the first few rows to verify
print("First few rows of the new dataset:")
print(df_hmdb51.head())

# Output the counts of videos in each split
train_videos = df_hmdb51[df_hmdb51['split'] == 'train']['filename'].count()
val_videos = df_hmdb51[df_hmdb51['split'] == 'val']['filename'].count()
test_videos = df_hmdb51[df_hmdb51['split'] == 'test']['filename'].count()

print(f"Total training videos: {train_videos}")
print(f"Total validation videos: {val_videos}")
print(f"Total testing videos: {test_videos}")


First few rows of the new dataset:
                                            filename      label  split
0  /kaggle/input/hmdbdata/hmdb51/kick_ball/Britne...  kick_ball  train
1  /kaggle/input/hmdbdata/hmdb51/kick_ball/Ribery...  kick_ball  train
2  /kaggle/input/hmdbdata/hmdb51/kick_ball/GOAL_2...  kick_ball  train
3  /kaggle/input/hmdbdata/hmdb51/kick_ball/Amazin...  kick_ball  train
4  /kaggle/input/hmdbdata/hmdb51/kick_ball/Nike_S...  kick_ball  train
Total training videos: 4714
Total validation videos: 1010
Total testing videos: 1042


In [ ]:
import pandas as pd

df = pd.read_csv('/kaggle/working/new_hmdb51.csv')
print("Unique classes:", df['label'].nunique())
print(sorted(df['label'].unique()))


In [5]:
# Define Transforms
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    # Note: Normalization will be handled by the ViT feature extractor
])


In [6]:
class HMDB51Dataset(Dataset):
    VIDEO_EXTENSION = '.avi'

    def __init__(self, df_hmdb51, transform=None, num_samples=16, frames_per_video=32):
        if 'filename' not in df_hmdb51.columns or 'label' not in df_hmdb51.columns:
            raise ValueError("DataFrame must contain 'filename' and 'label' columns.")
        
        valid_videos = []
        for _, row in df_hmdb51.iterrows():
            video_path = row['filename'].replace('\\', '/')  # Ensure path uses the correct separators
            if self._is_valid_video(video_path):
                valid_videos.append({'filename': video_path, 'label': row['label']})

        if not valid_videos:
            raise ValueError("No valid videos found. Please check your dataset.")

        self.dataset = pd.DataFrame(valid_videos)
        self.video_paths = self.dataset['filename'].tolist()
        self.labels = self.dataset['label'].tolist()
        self.transform = transform
        self.label_to_index = {label: idx for idx, label in enumerate(sorted(set(self.labels)))}
        self.num_samples = num_samples
        self.frames_per_video = frames_per_video

        # Initialize ViT Image Processor
        self.vit_image_processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k', do_rescale=False)

    def _is_valid_video(self, video_path):
        if not os.path.isfile(video_path):
            print(f"File does not exist: {video_path}")
            return False
        if not video_path.endswith(self.VIDEO_EXTENSION):
            print(f"Skipping unsupported video format: {video_path}")
            return False
        return True

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        cap = cv2.VideoCapture(video_path)

        if not cap.isOpened():
            raise ValueError(f"Error opening video file: {video_path}")

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        frame_indices = self._get_frame_indices(total_frames)
        frames = []

        for i in frame_indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ret, frame = cap.read()
            if not ret:
                continue  # Skip unreadable frames without printing errors

            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            if self.transform:
                frame = Image.fromarray(frame)  # Convert frame to PIL Image
                frame = self.transform(frame)  # Apply transforms
            
            # Preprocess frame using ViT Image Processor
            frame = self.vit_image_processor(images=frame, return_tensors='pt')['pixel_values'].squeeze(0)
            frames.append(frame)

        cap.release()

        if len(frames) == 0:
            print(f"Warning: No frames extracted from video: {video_path}")
            return torch.zeros((self.frames_per_video, 3, 224, 224)), 0  # Return dummy data in case of an error

        frames = torch.stack(frames)
        label_index = self.label_to_index[label]
        return frames, label_index

    def _get_frame_indices(self, total_frames):
        if total_frames < self.frames_per_video:
            # Pad with repeated frames if total_frames < frames_per_video
            return [i % total_frames for i in range(self.frames_per_video)]
        step = total_frames // self.frames_per_video
        return [i * step for i in range(self.frames_per_video)]


In [7]:
# # Define a collate function to pad frames
# def collate_fn(batch):
#     frames = []
#     labels = []
#     max_frames = max([sample[0].shape[0] for sample in batch])  # Find max number of frames in a batch

#     for sample in batch:
#         frame_tensor, label = sample
#         if frame_tensor.shape[0] < max_frames:
#             padding = torch.zeros(max_frames - frame_tensor.shape[0], *frame_tensor.shape[1:])
#             padded_tensor = torch.cat([frame_tensor, padding], dim=0)
#         else:
#             padded_tensor = frame_tensor

#         frames.append(padded_tensor)
#         labels.append(label)

#     frames = torch.stack(frames)  # Stack all frames
#     labels = torch.tensor(labels)  # Stack all labels into tensor

#     return frames, labels



In [8]:
# Initialize the Dataset
train_dataset = HMDB51Dataset(df_hmdb51[df_hmdb51['split'] == 'train'], transform=transform)
val_dataset = HMDB51Dataset(df_hmdb51[df_hmdb51['split'] == 'val'], transform=transform)
test_dataset = HMDB51Dataset(df_hmdb51[df_hmdb51['split'] == 'test'], transform=transform)

# Create DataLoader instances
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn, pin_memory=True)

# Example to check the data
for frames, labels in train_loader:
    print(f"Frames batch shape: {frames.shape}")
    print(f"Labels batch shape: {labels.shape}")
    break  # Check for one batch

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

/opt/conda/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Frames batch shape: torch.Size([4, 32, 3, 224, 224])
Labels batch shape: torch.Size([4])


/opt/conda/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


In [9]:
# Focal Loss Function
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        BCE_loss = nn.functional.cross_entropy(inputs, targets, reduction='none')
        targets_one_hot = nn.functional.one_hot(targets, num_classes=inputs.size(1))
        focal_loss = self.alpha * (1 - torch.exp(-BCE_loss)) ** self.gamma * BCE_loss

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss


In [10]:
# Define the ViT + GRU Model
class ViT_GRU_TimeS(nn.Module):
    def __init__(self, num_classes=51, hidden_size=512, num_layers=2, dropout=0.5):
        super(ViT_GRU_TimeS, self).__init__()
        # Load the Vision Transformer model
        self.vit = create_model('vit_small_patch16_224', pretrained=True)
        self.vit.head = nn.Linear(self.vit.head.in_features, 512)  # Output 512-dimensional feature vectors
        
        # Initialize GRU
        self.gru = nn.GRU(input_size=512, hidden_size=hidden_size, num_layers=num_layers, 
                          batch_first=True, dropout=dropout)
        
        # Classifier layers
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 1024),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        N, D, C, H, W = x.shape  # N: batch size, D: number of frames, C: channels, H: height, W: width
        
        # Initialize a list to hold features for each frame
        frame_features = []

        # Process each frame one by one
        for i in range(D):
            frame = x[:, i, :, :, :]  # Select the ith frame from all samples in the batch
            vit_features = self.vit(frame)  # Extract features using ViT
            frame_features.append(vit_features.unsqueeze(1))  # Append features for each frame, add a dimension

        # Stack the frame features along the time dimension
        frame_features = torch.cat(frame_features, dim=1)  # Shape: (N, D, 512)

        # Pass through the GRU
        gru_output, _ = self.gru(frame_features)

        # Use the last GRU output for classification
        last_gru_output = gru_output[:, -1, :]  # Get the last time step output
        outputs = self.classifier(last_gru_output)

        return outputs

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(train_dataset.label_to_index)
model = ViT_GRU_TimeS(num_classes=num_classes).to(device)

# Initialize optimizer with weight decay
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = FocalLoss()  # Use focal loss

Loading pretrained weights from Hugging Face hub (timm/vit_small_patch16_224.augreg_in21k_ft_in1k)


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

[timm/vit_small_patch16_224.augreg_in21k_ft_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.


In [10]:
class ViT_Only(nn.Module):
    def __init__(self, num_classes=51, dropout=0.5):
        super(ViT_Only, self).__init__()
        self.vit = create_model('vit_small_patch16_224', pretrained=True)
        self.vit.head = nn.Linear(self.vit.head.in_features, 512)

        self.classifier = nn.Sequential(
            nn.Linear(512, 1024),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        N, D, C, H, W = x.shape
        frame_features = []

        for i in range(D):
            frame = x[:, i, :, :, :]
            vit_features = self.vit(frame)  # Shape: (N, 512)
            frame_features.append(vit_features.unsqueeze(1))

        frame_features = torch.cat(frame_features, dim=1)  # Shape: (N, D, 512)
        avg_features = frame_features.mean(dim=1)  # Shape: (N, 512)

        return self.classifier(avg_features)
        
# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(train_dataset.label_to_index)
model = ViT_Only(num_classes=num_classes).to(device)

# Initialize optimizer with weight decay
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)
criterion = FocalLoss()  # Use focal loss

Loading pretrained weights from Hugging Face hub (timm/vit_small_patch16_224.augreg_in21k_ft_in1k)


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

[timm/vit_small_patch16_224.augreg_in21k_ft_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.


In [10]:
import torch
import torch.nn as nn
import torch.optim as optim

# Assume you use a ViT or CNN as a feature extractor
class GRU_With_FeatureExtractor(nn.Module):
    def __init__(self, feature_extractor, input_size=384, hidden_size=512, num_layers=2, num_classes=51, dropout=0.5):
        super(GRU_With_FeatureExtractor, self).__init__()
        self.feature_extractor = feature_extractor  # e.g., a CNN or ViT without classifier head
        self.feature_extractor.head = nn.Identity()  # Remove classifier head if needed

        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True, dropout=dropout)
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 1024),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        N, D, C, H, W = x.shape
        frame_features = []

        for i in range(D):
            frame = x[:, i, :, :, :]  # shape: (N, C, H, W)
            features = self.feature_extractor(frame)  # shape: (N, input_size)
            frame_features.append(features.unsqueeze(1))  # shape: (N, 1, input_size)

        features_seq = torch.cat(frame_features, dim=1)  # shape: (N, D, input_size)
        gru_output, _ = self.gru(features_seq)
        last_output = gru_output[:, -1, :]
        return self.classifier(last_output)



# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

vit = create_model('vit_small_patch16_224', pretrained=True)
model = GRU_With_FeatureExtractor(vit, input_size=384, num_classes=len(train_dataset.label_to_index)).to(device)

# Example: extract number of classes from dataset
num_classes = len(train_dataset.label_to_index)


# Optimizer and Loss
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

# Focal Loss (or change to CrossEntropyLoss if needed)
criterion = FocalLoss()


Loading pretrained weights from Hugging Face hub (timm/vit_small_patch16_224.augreg_in21k_ft_in1k)


model.safetensors:   0%|          | 0.00/88.2M [00:00<?, ?B/s]

[timm/vit_small_patch16_224.augreg_in21k_ft_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.


In [11]:
# Learning Rate Scheduler with Warmup
class WarmupCosineAnnealingLR(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, warmup_epochs, max_epochs, last_epoch=-1):
        self.warmup_epochs = warmup_epochs
        self.max_epochs = max_epochs
        self.initial_lr = [group['lr'] for group in optimizer.param_groups]
        self.warmup_factor = 1.0 / self.warmup_epochs
        super(WarmupCosineAnnealingLR, self).__init__(optimizer, last_epoch)

    def get_lr(self):
        if self.last_epoch < self.warmup_epochs:
            return [base_lr * (self.last_epoch + 1) * self.warmup_factor for base_lr in self.initial_lr]
        
        progress = (self.last_epoch - self.warmup_epochs) / (self.max_epochs - self.warmup_epochs)
        lr = [base_lr * (0.5 * (1.0 + np.cos(np.pi * progress))) for base_lr in self.initial_lr]
        return lr

# Instantiate the scheduler
scheduler = WarmupCosineAnnealingLR(optimizer, warmup_epochs=5, max_epochs=10)

In [12]:
# Step 7: Define Training and Evaluation Functions
def calculate_metrics(labels, preds):
    labels_np = labels.cpu().numpy()
    preds_np = preds.cpu().numpy()
    precision = precision_score(labels_np, preds_np, average='weighted') * 100
    recall = recall_score(labels_np, preds_np, average='weighted') * 100
    f1 = f1_score(labels_np, preds_np, average='weighted') * 100
    return precision, recall, f1

def train(model, dataloader, optimizer, criterion, device):
    model.train()  # Set the model to training mode
    total_loss = 0
    correct = 0
    total = 0

    for inputs, labels in tqdm(dataloader):
        inputs, labels = inputs.to(device), labels.to(device)  # Move to the correct device
        
        # Ensure labels are in the right format (if needed)
        if len(labels.shape) != 1:  # Check if labels are not 1D
            raise ValueError("Labels must be a 1D tensor.")
        
        optimizer.zero_grad()  # Zero the gradients
        outputs = model(inputs)  # Forward pass
        loss = criterion(outputs, labels)  # Compute loss
        loss.backward()  # Backward pass
        optimizer.step()  # Update parameters

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    return avg_loss, accuracy

def evaluate(model, dataloader, device):
    model.eval()  # Set the model to evaluation mode
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in tqdm(dataloader):
            inputs, labels = inputs.to(device), labels.to(device)  # Move to the correct device
            outputs = model(inputs)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss

            total_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(predicted.cpu().numpy())  # Store predictions
            all_labels.extend(labels.cpu().numpy())  # Store true labels

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    precision, recall, f1 = calculate_metrics(torch.tensor(all_labels), torch.tensor(all_preds))
    return avg_loss, accuracy, precision, recall, f1



In [ ]:
# Step 8: Train the Model
num_epochs = 10
best_val_acc = 0.0

for epoch in range(num_epochs):
    # Train the model and get metrics
    train_loss, train_acc = train(model, train_loader, optimizer, criterion, device)
    
    # Evaluate the model on the validation set
    val_loss, val_acc, val_precision, val_recall, val_f1 = evaluate(model, val_loader, device)
    
    # Step the scheduler
    scheduler.step()

    # Log the metrics for the current epoch
    print(f'Epoch [{epoch + 1}/{num_epochs}], '
                f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, '
                f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, '
                f'Val Precision: {val_precision:.4f}, Val Recall: {val_recall:.4f}, Val F1: {val_f1:.4f}')

    # Save the model if validation accuracy improves
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_hybrid_model_HMDB_ablation_GRU.pth')
        


In [ ]:
# Save model, optimizer state, and epoch number
torch.save({
    'epoch': 10,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
},  'GRU_model_hmdb51.pth')

print("Model, optimizer state, and epoch number saved successfully as 'viT_GRU_model_hmdb.pth'")

In [ ]:
def test(model, test_loader, criterion, device):
    model.eval()  # Set the model to evaluation mode
    y_true = []
    y_pred = []
    total_loss = 0.0  # Initialize total loss

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)  # Ensure data is on the correct device

            # Forward pass
            outputs = model(inputs)
            loss = criterion(outputs, labels)  # Compute loss for this batch
            total_loss += loss.item()  # Accumulate loss

            _, predicted = torch.max(outputs, 1)

            # Collect true and predicted labels for metrics calculation
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(predicted.cpu().numpy())

    # Calculate metrics
    avg_loss = total_loss / len(test_loader)  # Average test loss
    test_acc = accuracy_score(y_true, y_pred)
    test_precision = precision_score(y_true, y_pred, average='macro')
    test_recall = recall_score(y_true, y_pred, average='macro')
    test_f1 = f1_score(y_true, y_pred, average='macro')

    print(f'Test Loss: {avg_loss:.4f}, Test Accuracy: {test_acc:.4f}, Test Precision: {test_precision:.4f}, '
          f'Test Recall: {test_recall:.4f}, Test F1: {test_f1:.4f}')

    return avg_loss, test_acc, test_precision, test_recall, test_f1, y_true, y_pred

In [ ]:
# Assuming `model` is your trained model, `test_loader` is your test DataLoader
# Load the saved model state
# Load the full checkpoint
checkpoint = torch.load('/kaggle/input/vit_gru_hmdb51/pytorch/default/1/viT_GRU_model_hmdb51.pth', map_location=device)

# Load only the model weights
model.load_state_dict(checkpoint['model_state_dict'])
#model.load_state_dict(torch.load('/kaggle/input/ucf50model/pytorch/default/1/viT_GRU_model_ucf50.pth'))
model.to(device)
model.eval()  # Set the model to evaluation mode

# Run the test function and store the results
avg_loss, test_acc, test_precision, test_recall, test_f1, y_true, y_pred = test(model, test_loader, criterion, device)

In [ ]:
import os
def plot_confusion_matrix(y_true, y_pred, class_names=None, filename='/kaggle/working/Confusion_Matrix_HMDB51_ViT_GRU.png'):
    conf_mat = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(18, 16))  # Larger for better visibility
    sns.heatmap(conf_mat, annot=True, fmt="d", cmap="Blues", cbar=True,
                square=True, linewidths=0.3, linecolor='gray')

    plt.title("Confusion Matrix", fontsize=20)
    plt.xlabel("Predicted Class", fontsize=14)
    plt.ylabel("True Class", fontsize=14)

    num_classes = conf_mat.shape[0]
    ticks = np.arange(num_classes)

    # Use class names if provided, otherwise use index
    if class_names:
        plt.xticks(ticks=ticks, labels=class_names, rotation=90, fontsize=8)
        plt.yticks(ticks=ticks, labels=class_names, fontsize=8)
    else:
        plt.xticks(ticks=ticks, labels=[str(i) for i in range(num_classes)], rotation=90, fontsize=8)
        plt.yticks(ticks=ticks, labels=[str(i) for i in range(num_classes)], fontsize=8)

    plt.tight_layout()
    
    plt.show()

plot_confusion_matrix(y_true, y_pred, filename='/kaggle/working/Confusion_Matrix_HMDB51_ViT_GRU.png')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

def plot_class_wise_accuracy(y_true, y_pred, num_classes, filename='class_wise_accuracy.png'):
    # Generate confusion matrix
    conf_mat = confusion_matrix(y_true, y_pred)
    
    # Calculate class-wise accuracy
    class_accuracy = conf_mat.diagonal() / conf_mat.sum(axis=1) * 100  # Convert to percentage
    
    # Plotting the class-wise accuracy
    plt.figure(figsize=(14, 8))
    bars = plt.bar(range(num_classes), class_accuracy, color='red', edgecolor='black', width=0.8)

    # Customize the appearance to look similar to your uploaded example
    plt.xlabel("Classes", fontsize=14)
    plt.ylabel("Accuracy (%)", fontsize=14)
    plt.title("Class-wise Accuracy (%)", fontsize=16)
    plt.ylim(0, 100)  # Set y-axis to 0-100 for percentage scale
    plt.xticks(range(0, num_classes, 5), rotation=90, fontsize=10)  # Display every 5th class for readability

    # Add grid lines for better readability
    plt.grid(axis='y', linestyle='--', color='grey', alpha=0.7)

    # Save the plot
    plt.savefig(filename)  # Save the figure as a PNG file
    plt.close()  # Close the plot to free memory

# Call the function with your test results
num_classes = 51  # Set this to your actual number of classes
plot_class_wise_accuracy(y_true, y_pred, num_classes, filename='class_wise_accuracy.png')

# If you're using Kaggle, you can upload this file to your Kaggle notebook output directory.


In [ ]:
import pickle

results = {
    'y_true': y_true,
    'y_pred': y_pred,
    'avg_loss': avg_loss,
    'accuracy': test_acc,
    'precision': test_precision,
    'recall': test_recall,
    'f1_score': test_f1
}

with open('test_results.pkl', 'wb') as f:
    pickle.dump(results, f)

In [ ]:
print("Unique values in y_true:", sorted(set(y_true)))
print("Unique values in y_pred:", sorted(set(y_pred)))


In [ ]:
import pickle
with open('/kaggle/input/hmdb51results/test_results hmdb51.pkl', 'rb') as f:
    results = pickle.load(f)

# Extract the values
y_true = results['y_true']
y_pred = results['y_pred']
avg_loss = results['avg_loss']
accuracy = results['accuracy']
precision = results['precision']
recall = results['recall']
f1_score = results['f1_score']